In [1]:
import pandas as pd
import os

print("=" * 70)
print("AMPI CORRECTION — INPUT FILE CHECK")
print("=" * 70)

files = [
    "/content/UPI_Master_Dataset (1).csv",
    "/content/mm1_queuing_results.csv",
    "/content/sarima_residuals (1).csv"
]

for f in files:
    print(f"\n{f}")
    print("Exists:", os.path.exists(f))

    if os.path.exists(f):
        df = pd.read_csv(f)

        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())
        print("\nFirst 5 rows:")
        display(df.head())

print("\n" + "=" * 70)
print("INPUT FILE CHECK COMPLETE")
print("=" * 70)

AMPI CORRECTION — INPUT FILE CHECK

/content/UPI_Master_Dataset (1).csv
Exists: True
Shape: (125, 8)
Columns:
['month', 'upi_remitter_banks', 'total_volume_in_mn', 'td_pct', 'iss_mean_z', 'iss_max_z', 'stress_flag', 'max_bank']

First 5 rows:


,month,upi_remitter_banks,total_volume_in_mn,td_pct,iss_mean_z,iss_max_z,stress_flag,max_bank
0,2024-01,State Bank of India,3128.97,0.19,0.165913,1.330143,False,Union Bank of India
1,2024-01,HDFC Bank Ltd.,1040.25,0.00,0.165913,1.330143,False,Union Bank of India
2,2024-01,Bank of Baroda,781.72,0.04,0.165913,1.330143,False,Union Bank of India
3,2024-01,Union Bank of India,753.66,1.15,0.165913,1.330143,False,Union Bank of India
4,2024-01,Punjab National Bank,618.95,0.27,0.165913,1.330143,False,Union Bank of India



/content/mm1_queuing_results.csv
Exists: True
Shape: (66, 11)
Columns:
['month', 'volume_mn', 'value_cr', 'days_in_month', 'seconds_in_month', 'total_transactions', 'lambda_avg_tps', 'rho_floor', 'rho_sens_low', 'rho_sens_high', 'rho_peak_baseline']

First 5 rows:


,month,volume_mn,value_cr,days_in_month,seconds_in_month,total_transactions,lambda_avg_tps,rho_floor,rho_sens_low,rho_sens_high,rho_peak_baseline
0,2021-01-01,2302.73,431181.89,31,2678400,2.302730e+09,859.740890,0.124540,0.103783,0.069189,0.098100
1,2021-02-01,2292.90,425062.76,28,2419200,2.292900e+09,947.792659,0.137295,0.114413,0.076275,0.108147
2,2021-03-01,2731.68,504886.44,31,2678400,2.731680e+09,1019.892473,0.147739,0.123116,0.082077,0.116374
3,2021-04-01,2641.06,493663.68,30,2592000,2.641060e+09,1018.927469,0.147600,0.123000,0.082000,0.116264
4,2021-05-01,2539.57,490638.65,31,2678400,2.539570e+09,948.166816,0.137349,0.114458,0.076305,0.108190



/content/sarima_residuals (1).csv
Exists: True
Shape: (47, 2)
Columns:
['month', 'sarima_resid_z']

First 5 rows:


,month,sarima_resid_z
0,2022-02-01,-0.942194
1,2022-03-01,0.878326
2,2022-04-01,1.098041
3,2022-05-01,1.908578
4,2022-06-01,-0.452779



INPUT FILE CHECK COMPLETE


In [2]:
# ================================================================
# AMPI CORRECTION — REBUILD 24-MONTH AMPI WITH CORRECTED SARIMA
# ================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("AMPI CORRECTION — 24-MONTH REBUILD")
print("=" * 70)

# ------------------------------------------------
# 1. LOAD THE THREE AUTHORITATIVE INPUT FILES
# ------------------------------------------------

upi_path = "/content/UPI_Master_Dataset (1).csv"
sarima_path = "/content/sarima_residuals (1).csv"
rho_path = "/content/mm1_queuing_results.csv"

upi = pd.read_csv(upi_path)
sarima = pd.read_csv(sarima_path)
rho = pd.read_csv(rho_path)

# Convert month columns to datetime
upi["month"] = pd.to_datetime(upi["month"])
sarima["month"] = pd.to_datetime(sarima["month"])
rho["month"] = pd.to_datetime(rho["month"])

print("\nFiles loaded successfully.")
print("UPI:", upi.shape)
print("SARIMA:", sarima.shape)
print("RHO:", rho.shape)

# ------------------------------------------------
# 2. DEDUPLICATE UPI TO ONE ROW PER MONTH
# ------------------------------------------------

upi_monthly = (
    upi[
        ["month", "iss_mean_z", "iss_max_z"]
    ]
    .drop_duplicates(subset=["month"])
    .copy()
)

# ------------------------------------------------
# 3. KEEP CORRECTED SARIMA SERIES
# ------------------------------------------------

sarima_monthly = (
    sarima[
        ["month", "sarima_resid_z"]
    ]
    .drop_duplicates(subset=["month"])
    .copy()
)

# ------------------------------------------------
# 4. KEEP RHO PEAK BASELINE
# ------------------------------------------------

rho_monthly = (
    rho[
        ["month", "rho_peak_baseline"]
    ]
    .drop_duplicates(subset=["month"])
    .copy()
)

# ------------------------------------------------
# 5. MERGE ALL THREE SERIES
# ------------------------------------------------

merged = (
    upi_monthly
    .merge(sarima_monthly, on="month", how="inner")
    .merge(rho_monthly, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

# ------------------------------------------------
# 6. SELECT THE 24-MONTH COMMON WINDOW
# ------------------------------------------------

merged = merged.tail(24).copy()
merged = merged.sort_values("month").reset_index(drop=True)

print("\n" + "=" * 70)
print("24-MONTH CORRECTED AMPI INPUT")
print("=" * 70)

print("Rows:", len(merged))
print("First month:", merged["month"].min())
print("Last month :", merged["month"].max())
print("Duplicate months:", merged["month"].duplicated().sum())

print("\nMissing values:")
print(merged.isna().sum())

# ------------------------------------------------
# 7. VERIFY CORRECTED SARIMA VALUE
# ------------------------------------------------

feb_2022 = sarima[sarima["month"] == pd.Timestamp("2022-02-01")]

print("\n" + "=" * 70)
print("SARIMA CORRECTION CHECK")
print("=" * 70)

if len(feb_2022) > 0:
    print(
        "2022-02-01 sarima_resid_z:",
        feb_2022.iloc[0]["sarima_resid_z"]
    )
    print("Expected corrected value: -0.942194")

# ------------------------------------------------
# 8. NORMALIZE EACH COMPONENT TO 70–130
# ------------------------------------------------

def normalize_70_130(series):
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return pd.Series(100.0, index=series.index)

    return 70 + 60 * (series - min_val) / (max_val - min_val)


merged["iss_mean_norm"] = normalize_70_130(
    merged["iss_mean_z"]
)

merged["sarima_norm"] = normalize_70_130(
    merged["sarima_resid_z"]
)

merged["rho_norm"] = normalize_70_130(
    merged["rho_peak_baseline"]
)

# ------------------------------------------------
# 9. CALCULATE AMPI
# ------------------------------------------------

merged["MeanPopulation_SD"] = (
    merged[
        ["iss_mean_norm", "sarima_norm", "rho_norm"]
    ].mean(axis=1)
)

merged["SD"] = (
    merged[
        ["iss_mean_norm", "sarima_norm", "rho_norm"]
    ].std(axis=1, ddof=1)
)

merged["CV"] = (
    merged["SD"] / merged["MeanPopulation_SD"]
)

# Official CV penalty formula used previously
merged["CVPenalty"] = (
    merged["MeanPopulation_SD"] * merged["CV"]
)

merged["AMPI_final"] = (
    merged["MeanPopulation_SD"] - merged["CVPenalty"]
)

# ------------------------------------------------
# 10. NORMALIZATION VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("NORMALIZATION CHECK")
print("=" * 70)

for col in ["iss_mean_norm", "sarima_norm", "rho_norm"]:
    print(
        f"{col}: min = {merged[col].min():.6f}, "
        f"max = {merged[col].max():.6f}"
    )

# ------------------------------------------------
# 11. FORMULA VALIDATION
# ------------------------------------------------

formula_check = (
    merged["MeanPopulation_SD"]
    - merged["CVPenalty"]
)

formula_difference = np.max(
    np.abs(formula_check - merged["AMPI_final"])
)

print("\n" + "=" * 70)
print("AMPI FORMULA VALIDATION")
print("=" * 70)

print(
    f"Maximum formula difference: "
    f"{formula_difference:.12f}"
)

# ------------------------------------------------
# 12. FINAL AMPI STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED AMPI STATISTICS")
print("=" * 70)

print("Observations:", len(merged))
print(f"Minimum : {merged['AMPI_final'].min():.4f}")
print(f"Maximum : {merged['AMPI_final'].max():.4f}")
print(f"Mean    : {merged['AMPI_final'].mean():.4f}")
print(f"Median  : {merged['AMPI_final'].median():.4f}")

# ------------------------------------------------
# 13. FULL 24-MONTH AMPI TABLE
# ------------------------------------------------

print("\n" + "=" * 70)
print("FULL 24-MONTH CORRECTED AMPI SERIES")
print("=" * 70)

display(
    merged[
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "MeanPopulation_SD",
            "SD",
            "CV",
            "CVPenalty",
            "AMPI_final"
        ]
    ].round(6)
)

# ------------------------------------------------
# 14. MARCH 2025 / APRIL 2025 CHECK
# ------------------------------------------------

# Rank: highest AMPI = rank 1
merged["AMPI_rank"] = (
    merged["AMPI_final"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Percentile: percentage of months at or below this AMPI
merged["AMPI_percentile"] = (
    merged["AMPI_final"]
    .rank(method="average", pct=True)
    * 100
)

target_months = merged[
    merged["month"].isin(
        [
            pd.Timestamp("2025-03-01"),
            pd.Timestamp("2025-04-01")
        ]
    )
].copy()

print("\n" + "=" * 70)
print("PRE-REGISTERED OUTAGE MONTH CHECK")
print("=" * 70)

display(
    target_months[
        [
            "month",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "CVPenalty"
        ]
    ].round(6)
)

# ------------------------------------------------
# 15. TOP / BOTTOM MONTH
# ------------------------------------------------

top_row = merged.loc[
    merged["AMPI_final"].idxmax()
]

bottom_row = merged.loc[
    merged["AMPI_final"].idxmin()
]

print("\n" + "=" * 70)
print("HIGHEST / LOWEST AMPI")
print("=" * 70)

print(
    f"Highest: {top_row['month'].date()} "
    f"| AMPI = {top_row['AMPI_final']:.4f}"
)

print(
    f"Lowest : {bottom_row['month'].date()} "
    f"| AMPI = {bottom_row['AMPI_final']:.4f}"
)

# ------------------------------------------------
# 16. SAVE CORRECTED OUTPUT
# ------------------------------------------------

merged.to_csv(
    "/content/AMPI_24_month_corrected_sarima.csv",
    index=False
)

merged[
    [
        "month",
        "AMPI_final",
        "AMPI_rank",
        "AMPI_percentile"
    ]
].to_csv(
    "/content/AMPI_24_month_corrected_sarima_summary.csv",
    index=False
)

print("\n" + "=" * 70)
print("CORRECTED AMPI REBUILD COMPLETE")
print("=" * 70)

print("\nSaved:")
print("/content/AMPI_24_month_corrected_sarima.csv")
print("/content/AMPI_24_month_corrected_sarima_summary.csv")

AMPI CORRECTION — 24-MONTH REBUILD

Files loaded successfully.
UPI: (125, 8)
SARIMA: (47, 2)
RHO: (66, 11)

24-MONTH CORRECTED AMPI INPUT
Rows: 24
First month: 2024-01-01 00:00:00
Last month : 2025-12-01 00:00:00
Duplicate months: 0

Missing values:
month                0
iss_mean_z           0
iss_max_z            0
sarima_resid_z       0
rho_peak_baseline    0
dtype: int64

SARIMA CORRECTION CHECK
2022-02-01 sarima_resid_z: -0.9421944543974712
Expected corrected value: -0.942194

NORMALIZATION CHECK
iss_mean_norm: min = 70.000000, max = 130.000000
sarima_norm: min = 70.000000, max = 130.000000
rho_norm: min = 70.000000, max = 130.000000

AMPI FORMULA VALIDATION
Maximum formula difference: 0.000000000000

CORRECTED AMPI STATISTICS
Observations: 24
Minimum : 60.8810
Maximum : 88.9205
Mean    : 76.3340
Median  : 75.6339

FULL 24-MONTH CORRECTED AMPI SERIES


,month,iss_mean_norm,sarima_norm,rho_norm,MeanPopulation_SD,SD,CV,CVPenalty,AMPI_final
0,2024-01-01,83.495102,103.369885,70.000000,85.621662,16.786274,0.196052,16.786274,68.835388
1,2024-02-01,103.445385,115.736065,74.671400,97.950950,21.076486,0.215174,21.076486,76.874464
2,2024-03-01,91.384883,116.259516,77.869122,95.171174,19.473253,0.204613,19.473253,75.697920
3,2024-04-01,130.000000,93.732736,79.825023,101.185920,25.904527,0.256009,25.904527,75.281392
4,2024-05-01,87.446533,104.768291,81.659593,91.291472,12.024585,0.131716,12.024585,79.266887
5,2024-06-01,74.829032,98.203755,83.645277,85.559355,11.804329,0.137967,11.804329,73.755026
6,2024-07-01,76.219596,95.757964,84.202372,85.393310,9.823477,0.115038,9.823477,75.569833
7,2024-08-01,90.595480,94.547818,87.558094,90.900464,3.504828,0.038557,3.504828,87.395636
8,2024-09-01,73.265122,99.925778,91.248382,88.146427,13.598318,0.154270,13.598318,74.548110
9,2024-10-01,70.517691,130.000000,97.876034,99.464575,29.772955,0.299332,29.772955,69.691620



PRE-REGISTERED OUTAGE MONTH CHECK


,month,AMPI_final,AMPI_rank,AMPI_percentile,iss_mean_norm,sarima_norm,rho_norm,CVPenalty
14,2025-03-01,81.811354,6,79.166667,79.651774,122.030511,108.795905,21.681377
15,2025-04-01,79.030330,8,70.833333,78.248056,96.672548,109.994152,15.941255



HIGHEST / LOWEST AMPI
Highest: 2025-01-01 | AMPI = 88.9205
Lowest : 2025-10-01 | AMPI = 60.8810

CORRECTED AMPI REBUILD COMPLETE

Saved:
/content/AMPI_24_month_corrected_sarima.csv
/content/AMPI_24_month_corrected_sarima_summary.csv


In [3]:
# ================================================================
# AMPI CORRECTION — OFFICIAL CV PENALTY FORMULA
# ================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("AMPI CORRECTION — OFFICIAL FORMULA")
print("=" * 70)

# ------------------------------------------------
# 1. LOAD THE CORRECTED-SARIMA AMPI INPUT
# ------------------------------------------------

input_file = "/content/AMPI_24_month_corrected_sarima.csv"

df = pd.read_csv(input_file)
df["month"] = pd.to_datetime(df["month"])

print("\nInput loaded:")
print("Rows:", len(df))
print("First month:", df["month"].min())
print("Last month :", df["month"].max())

# ------------------------------------------------
# 2. RE-CALCULATE THE OFFICIAL AMPI COMPONENTS
# ------------------------------------------------

components = [
    "iss_mean_norm",
    "sarima_norm",
    "rho_norm"
]

# Mean of the three normalized components
df["MeanPopulation_SD"] = df[components].mean(axis=1)

# Sample standard deviation, same as official calculation
df["SD"] = df[components].std(axis=1, ddof=1)

# Coefficient of variation
df["CV"] = (
    df["SD"] / df["MeanPopulation_SD"]
)

# OFFICIAL CV PENALTY:
# CVPenalty = SD × CV
df["CVPenalty"] = (
    df["SD"] * df["CV"]
)

# OFFICIAL AMPI:
# AMPI = MeanPopulation_SD − CVPenalty
df["AMPI_final"] = (
    df["MeanPopulation_SD"]
    - df["CVPenalty"]
)

# ------------------------------------------------
# 3. FORMULA VALIDATION
# ------------------------------------------------

independent_ampi = (
    df["MeanPopulation_SD"]
    - (
        df["SD"]
        * (
            df["SD"]
            / df["MeanPopulation_SD"]
        )
    )
)

formula_difference = np.max(
    np.abs(
        df["AMPI_final"]
        - independent_ampi
    )
)

print("\n" + "=" * 70)
print("OFFICIAL FORMULA VALIDATION")
print("=" * 70)

print(
    f"Maximum formula difference: "
    f"{formula_difference:.12f}"
)

# ------------------------------------------------
# 4. FINAL STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED AMPI STATISTICS")
print("=" * 70)

print("Observations:", len(df))
print(f"Minimum : {df['AMPI_final'].min():.4f}")
print(f"Maximum : {df['AMPI_final'].max():.4f}")
print(f"Mean    : {df['AMPI_final'].mean():.4f}")
print(f"Median  : {df['AMPI_final'].median():.4f}")

# ------------------------------------------------
# 5. RANK ALL 24 MONTHS
# ------------------------------------------------

df["AMPI_rank"] = (
    df["AMPI_final"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df["AMPI_percentile"] = (
    df["AMPI_final"]
    .rank(method="average", pct=True)
    * 100
)

# ------------------------------------------------
# 6. FULL 24-MONTH AMPI SERIES
# ------------------------------------------------

print("\n" + "=" * 70)
print("FULL 24-MONTH CORRECTED AMPI SERIES")
print("=" * 70)

display(
    df[
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "MeanPopulation_SD",
            "SD",
            "CV",
            "CVPenalty",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile"
        ]
    ].round(6)
)

# ------------------------------------------------
# 7. MARCH / APRIL 2025 PRE-REGISTERED CHECK
# ------------------------------------------------

outage_months = df[
    df["month"].isin(
        [
            pd.Timestamp("2025-03-01"),
            pd.Timestamp("2025-04-01")
        ]
    )
].copy()

print("\n" + "=" * 70)
print("PRE-REGISTERED OUTAGE MONTH CHECK")
print("=" * 70)

display(
    outage_months[
        [
            "month",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "CVPenalty"
        ]
    ].round(6)
)

# ------------------------------------------------
# 8. HIGHEST / LOWEST AMPI
# ------------------------------------------------

highest = df.loc[df["AMPI_final"].idxmax()]
lowest = df.loc[df["AMPI_final"].idxmin()]

print("\n" + "=" * 70)
print("HIGHEST / LOWEST AMPI")
print("=" * 70)

print(
    f"Highest: {highest['month'].date()} "
    f"| AMPI = {highest['AMPI_final']:.4f}"
)

print(
    f"Lowest : {lowest['month'].date()} "
    f"| AMPI = {lowest['AMPI_final']:.4f}"
)

# ------------------------------------------------
# 9. SAVE FINAL CORRECTED FILES
# ------------------------------------------------

final_file = "/content/AMPI_24_month_FINAL_corrected_sarima.csv"
summary_file = "/content/AMPI_24_month_FINAL_corrected_sarima_summary.csv"

df.to_csv(final_file, index=False)

df[
    [
        "month",
        "AMPI_final",
        "AMPI_rank",
        "AMPI_percentile"
    ]
].to_csv(summary_file, index=False)

print("\n" + "=" * 70)
print("FINAL CORRECTED AMPI FILES SAVED")
print("=" * 70)

print(final_file)
print(summary_file)

print("\n" + "=" * 70)
print("AMPI CORRECTION COMPLETE")
print("=" * 70)

AMPI CORRECTION — OFFICIAL FORMULA

Input loaded:
Rows: 24
First month: 2024-01-01 00:00:00
Last month : 2025-12-01 00:00:00

OFFICIAL FORMULA VALIDATION
Maximum formula difference: 0.000000000000

CORRECTED AMPI STATISTICS
Observations: 24
Minimum : 78.0929
Maximum : 101.2355
Mean    : 90.6808
Median  : 90.9760

FULL 24-MONTH CORRECTED AMPI SERIES


,month,iss_mean_norm,sarima_norm,rho_norm,MeanPopulation_SD,SD,CV,CVPenalty,AMPI_final,AMPI_rank,AMPI_percentile
0,2024-01-01,83.495102,103.369885,70.000000,85.621662,16.786274,0.196052,3.290978,82.330684,22,12.500000
1,2024-02-01,103.445385,115.736065,74.671400,97.950950,21.076486,0.215174,4.535109,93.415841,9,66.666667
2,2024-03-01,91.384883,116.259516,77.869122,95.171174,19.473253,0.204613,3.984480,91.186694,12,54.166667
3,2024-04-01,130.000000,93.732736,79.825023,101.185920,25.904527,0.256009,6.631798,94.554122,8,70.833333
4,2024-05-01,87.446533,104.768291,81.659593,91.291472,12.024585,0.131716,1.583835,89.707637,16,37.500000
5,2024-06-01,74.829032,98.203755,83.645277,85.559355,11.804329,0.137967,1.628603,83.930752,20,20.833333
6,2024-07-01,76.219596,95.757964,84.202372,85.393310,9.823477,0.115038,1.130073,84.263237,19,25.000000
7,2024-08-01,90.595480,94.547818,87.558094,90.900464,3.504828,0.038557,0.135135,90.765329,13,50.000000
8,2024-09-01,73.265122,99.925778,91.248382,88.146427,13.598318,0.154270,2.097808,86.048620,17,33.333333
9,2024-10-01,70.517691,130.000000,97.876034,99.464575,29.772955,0.299332,8.912006,90.552570,14,45.833333



PRE-REGISTERED OUTAGE MONTH CHECK


,month,AMPI_final,AMPI_rank,AMPI_percentile,iss_mean_norm,sarima_norm,rho_norm,CVPenalty
14,2025-03-01,98.950555,3,91.666667,79.651774,122.030511,108.795905,4.542175
15,2025-04-01,92.295799,11,58.333333,78.248056,96.672548,109.994152,2.675786



HIGHEST / LOWEST AMPI
Highest: 2025-12-01 | AMPI = 101.2355
Lowest : 2025-02-01 | AMPI = 78.0929

FINAL CORRECTED AMPI FILES SAVED
/content/AMPI_24_month_FINAL_corrected_sarima.csv
/content/AMPI_24_month_FINAL_corrected_sarima_summary.csv

AMPI CORRECTION COMPLETE


In [4]:
# ================================================================
# AMPI CORRECTION — DETRENDED RHO DIAGNOSTIC
# ================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("AMPI CORRECTION — DETRENDED RHO")
print("=" * 70)

# ------------------------------------------------
# 1. Load M/M/1 queuing results
# ------------------------------------------------
rho_file = "/content/mm1_queuing_results.csv"

rho_df = pd.read_csv(rho_file)
rho_df["month"] = pd.to_datetime(rho_df["month"])

print("\nInput:")
print("Rows:", len(rho_df))
print("Columns:", list(rho_df.columns))

# ------------------------------------------------
# 2. Keep the rho_peak_baseline series
# ------------------------------------------------
rho = rho_df[["month", "rho_peak_baseline"]].copy()
rho = rho.sort_values("month").reset_index(drop=True)

# ------------------------------------------------
# 3. Local-trend detrending
#
# Actual rho - locally predicted rho
#
# Rolling local mean is used as the local trend.
# Centered window = 5 months.
# ------------------------------------------------
window = 5

rho["rho_local_trend"] = (
    rho["rho_peak_baseline"]
    .rolling(window=window, center=True, min_periods=1)
    .mean()
)

rho["rho_detrended"] = (
    rho["rho_peak_baseline"] - rho["rho_local_trend"]
)

# ------------------------------------------------
# 4. Standardise detrended rho
# ------------------------------------------------
rho_mean = rho["rho_detrended"].mean()
rho_sd = rho["rho_detrended"].std(ddof=1)

rho["rho_detrended_z"] = (
    (rho["rho_detrended"] - rho_mean) / rho_sd
)

# ------------------------------------------------
# 5. Normalise to 70–130
# ------------------------------------------------
z_min = rho["rho_detrended_z"].min()
z_max = rho["rho_detrended_z"].max()

rho["rho_detrended_norm"] = (
    70
    + 60 * (
        (rho["rho_detrended_z"] - z_min)
        / (z_max - z_min)
    )
)

# ------------------------------------------------
# 6. Check remaining time trend
# ------------------------------------------------
rho["month_order"] = np.arange(1, len(rho) + 1)

time_corr = rho["month_order"].corr(
    rho["rho_detrended_norm"]
)

print("\n" + "=" * 70)
print("DETRENDING RESULTS")
print("=" * 70)

print("\nOriginal rho vs month order:")
print(
    rho["month_order"].corr(
        rho["rho_peak_baseline"]
    )
)

print("\nDetrended rho vs month order:")
print(time_corr)

print("\nDetrended rho statistics:")
print("Minimum:", rho["rho_detrended"].min())
print("Maximum:", rho["rho_detrended"].max())
print("Mean   :", rho["rho_detrended"].mean())
print("SD     :", rho["rho_detrended"].std(ddof=1))

print("\nNormalised detrended rho:")
print(
    "Minimum:",
    rho["rho_detrended_norm"].min()
)
print(
    "Maximum:",
    rho["rho_detrended_norm"].max()
)

# ------------------------------------------------
# 7. Show relevant period
# ------------------------------------------------
print("\n" + "=" * 70)
print("2024–2025 DETRENDED RHO")
print("=" * 70)

print(
    rho[
        (rho["month"] >= "2024-01-01") &
        (rho["month"] <= "2025-12-01")
    ][
        [
            "month",
            "rho_peak_baseline",
            "rho_local_trend",
            "rho_detrended",
            "rho_detrended_z",
            "rho_detrended_norm"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 8. Save diagnostic file
# ------------------------------------------------
output_file = "/content/rho_detrended_diagnostic.csv"

rho.to_csv(output_file, index=False)

print("\n" + "=" * 70)
print("DETRENDING DIAGNOSTIC COMPLETE")
print("=" * 70)
print("\nSaved:")
print(output_file)

AMPI CORRECTION — DETRENDED RHO

Input:
Rows: 66
Columns: ['month', 'volume_mn', 'value_cr', 'days_in_month', 'seconds_in_month', 'total_transactions', 'lambda_avg_tps', 'rho_floor', 'rho_sens_low', 'rho_sens_high', 'rho_peak_baseline']

DETRENDING RESULTS

Original rho vs month order:
0.9952289490850721

Detrended rho vs month order:
0.14881264910073289

Detrended rho statistics:
Minimum: -0.01586291256621708
Maximum: 0.026460656664302418
Mean   : 7.1089398785245335e-06
SD     : 0.006318582050164443

Normalised detrended rho:
Minimum: 70.0
Maximum: 130.0

2024–2025 DETRENDED RHO
     month  rho_peak_baseline  rho_local_trend  rho_detrended  rho_detrended_z  rho_detrended_norm
2024-01-01           0.519869         0.530053      -0.010184        -1.612916           78.050377
2024-02-01           0.551152         0.548266       0.002885         0.455518           96.578449
2024-03-01           0.572566         0.565440       0.007126         1.126652          102.590156
2024-04-01       

In [5]:
# ================================================================
# AMPI CORRECTION — REBUILD USING DETRENDED RHO
# ================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("AMPI CORRECTION — DETRENDED RHO REBUILD")
print("=" * 70)

# ------------------------------------------------
# 1. Load corrected input files
# ------------------------------------------------
upi = pd.read_csv("/content/UPI_Master_Dataset (1).csv")
sarima = pd.read_csv("/content/sarima_residuals (1).csv")
rho = pd.read_csv("/content/rho_detrended_diagnostic.csv")

upi["month"] = pd.to_datetime(upi["month"])
sarima["month"] = pd.to_datetime(sarima["month"])
rho["month"] = pd.to_datetime(rho["month"])

# ------------------------------------------------
# 2. One ISS observation per month
# ------------------------------------------------
iss = (
    upi.groupby("month", as_index=False)
       .agg({
           "iss_mean_z": "first",
           "iss_max_z": "first"
       })
)

# ------------------------------------------------
# 3. Keep corrected SARIMA
# ------------------------------------------------
sarima = sarima[["month", "sarima_resid_z"]].copy()

# ------------------------------------------------
# 4. Keep DETRENDED rho
# ------------------------------------------------
rho = rho[
    ["month", "rho_detrended_norm"]
].copy()

rho = rho.rename(
    columns={
        "rho_detrended_norm": "rho_norm"
    }
)

# ------------------------------------------------
# 5. Merge
# ------------------------------------------------
merged = (
    iss
    .merge(sarima, on="month", how="inner")
    .merge(rho, on="month", how="inner")
)

# ------------------------------------------------
# 6. Restrict to 24-month AMPI window
# ------------------------------------------------
merged = merged[
    (merged["month"] >= "2024-01-01") &
    (merged["month"] <= "2025-12-01")
].copy()

merged = merged.sort_values("month").reset_index(drop=True)

print("\n" + "=" * 70)
print("24-MONTH DETRENDED-RHO INPUT")
print("=" * 70)

print("Rows:", len(merged))
print("First month:", merged["month"].min())
print("Last month :", merged["month"].max())
print("Duplicate months:", merged["month"].duplicated().sum())

print("\nMissing values:")
print(merged.isna().sum())

# ------------------------------------------------
# 7. Normalize ISS
# ------------------------------------------------
def normalize_70_130(series):
    s_min = series.min()
    s_max = series.max()

    return 70 + 60 * (
        (series - s_min) / (s_max - s_min)
    )

merged["iss_mean_norm"] = normalize_70_130(
    merged["iss_mean_z"]
)

# ------------------------------------------------
# 8. Normalize corrected SARIMA
# ------------------------------------------------
merged["sarima_norm"] = normalize_70_130(
    merged["sarima_resid_z"]
)

# ------------------------------------------------
# 9. rho is ALREADY normalized 70–130
# ------------------------------------------------
merged["rho_norm"] = merged["rho_norm"]

print("\n" + "=" * 70)
print("NORMALIZATION CHECK")
print("=" * 70)

print(
    "iss_mean_norm:",
    merged["iss_mean_norm"].min(),
    "to",
    merged["iss_mean_norm"].max()
)

print(
    "sarima_norm:",
    merged["sarima_norm"].min(),
    "to",
    merged["sarima_norm"].max()
)

print(
    "rho_norm:",
    merged["rho_norm"].min(),
    "to",
    merged["rho_norm"].max()
)

# ------------------------------------------------
# 10. OFFICIAL AMPI FORMULA
# ------------------------------------------------
merged["MeanPopulation_SD"] = (
    merged[
        [
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm"
        ]
    ].mean(axis=1)
)

merged["SD"] = (
    merged[
        [
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm"
        ]
    ].std(axis=1, ddof=1)
)

merged["CV"] = (
    merged["SD"] /
    merged["MeanPopulation_SD"]
)

# Official penalty = Mean × CV²
merged["CVPenalty"] = (
    merged["MeanPopulation_SD"] *
    merged["CV"] ** 2
)

merged["AMPI_final"] = (
    merged["MeanPopulation_SD"]
    - merged["CVPenalty"]
)

# ------------------------------------------------
# 11. Formula validation
# ------------------------------------------------
independent_ampi = (
    merged["MeanPopulation_SD"]
    - (
        merged["MeanPopulation_SD"]
        * merged["CV"] ** 2
    )
)

formula_difference = np.max(
    np.abs(
        merged["AMPI_final"]
        - independent_ampi
    )
)

print("\n" + "=" * 70)
print("AMPI FORMULA VALIDATION")
print("=" * 70)

print(
    "Maximum formula difference:",
    f"{formula_difference:.12f}"
)

# ------------------------------------------------
# 12. Rank AMPI
# ------------------------------------------------
merged["AMPI_rank"] = (
    merged["AMPI_final"]
    .rank(method="min", ascending=False)
    .astype(int)
)

merged["AMPI_percentile"] = (
    (len(merged) - merged["AMPI_rank"])
    / (len(merged) - 1)
    * 100
)

# ------------------------------------------------
# 13. Statistics
# ------------------------------------------------
print("\n" + "=" * 70)
print("DE-TRENDED RHO AMPI STATISTICS")
print("=" * 70)

print("Observations:", len(merged))
print("Minimum :", f"{merged['AMPI_final'].min():.4f}")
print("Maximum :", f"{merged['AMPI_final'].max():.4f}")
print("Mean    :", f"{merged['AMPI_final'].mean():.4f}")
print("Median  :", f"{merged['AMPI_final'].median():.4f}")

# ------------------------------------------------
# 14. Full series
# ------------------------------------------------
print("\n" + "=" * 70)
print("FULL 24-MONTH AMPI — DETRENDED RHO")
print("=" * 70)

print(
    merged[
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "MeanPopulation_SD",
            "SD",
            "CV",
            "CVPenalty",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 15. PRE-REGISTERED OUTAGE MONTH CHECK
# ------------------------------------------------
print("\n" + "=" * 70)
print("PRE-REGISTERED OUTAGE MONTH CHECK")
print("=" * 70)

outage = merged[
    merged["month"].isin(
        pd.to_datetime(
            ["2025-03-01", "2025-04-01"]
        )
    )
]

print(
    outage[
        [
            "month",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "CVPenalty"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 16. Highest / lowest
# ------------------------------------------------
highest = merged.loc[
    merged["AMPI_final"].idxmax()
]

lowest = merged.loc[
    merged["AMPI_final"].idxmin()
]

print("\n" + "=" * 70)
print("HIGHEST / LOWEST AMPI")
print("=" * 70)

print(
    f"Highest: {highest['month'].date()} | "
    f"AMPI = {highest['AMPI_final']:.4f}"
)

print(
    f"Lowest : {lowest['month'].date()} | "
    f"AMPI = {lowest['AMPI_final']:.4f}"
)

# ------------------------------------------------
# 17. Save corrected output
# ------------------------------------------------
output_file = (
    "/content/AMPI_24_month_DETRENDED_RHO.csv"
)

summary_file = (
    "/content/AMPI_24_month_DETRENDED_RHO_summary.csv"
)

merged.to_csv(output_file, index=False)

summary = pd.DataFrame({
    "metric": [
        "observations",
        "minimum",
        "maximum",
        "mean",
        "median",
        "formula_difference",
        "march_2025_rank",
        "march_2025_percentile",
        "april_2025_rank",
        "april_2025_percentile"
    ],
    "value": [
        len(merged),
        merged["AMPI_final"].min(),
        merged["AMPI_final"].max(),
        merged["AMPI_final"].mean(),
        merged["AMPI_final"].median(),
        formula_difference,
        outage.loc[
            outage["month"] == "2025-03-01",
            "AMPI_rank"
        ].iloc[0],
        outage.loc[
            outage["month"] == "2025-03-01",
            "AMPI_percentile"
        ].iloc[0],
        outage.loc[
            outage["month"] == "2025-04-01",
            "AMPI_rank"
        ].iloc[0],
        outage.loc[
            outage["month"] == "2025-04-01",
            "AMPI_percentile"
        ].iloc[0]
    ]
})

summary.to_csv(summary_file, index=False)

print("\n" + "=" * 70)
print("DE-TRENDED RHO AMPI REBUILD COMPLETE")
print("=" * 70)

print("\nSaved:")
print(output_file)
print(summary_file)

AMPI CORRECTION — DETRENDED RHO REBUILD

24-MONTH DETRENDED-RHO INPUT
Rows: 24
First month: 2024-01-01 00:00:00
Last month : 2025-12-01 00:00:00
Duplicate months: 0

Missing values:
month             0
iss_mean_z        0
iss_max_z         0
sarima_resid_z    0
rho_norm          0
dtype: int64

NORMALIZATION CHECK
iss_mean_norm: 70.0 to 130.0
sarima_norm: 70.0 to 130.0
rho_norm: 70.0 to 130.0

AMPI FORMULA VALIDATION
Maximum formula difference: 0.000000000000

DE-TRENDED RHO AMPI STATISTICS
Observations: 24
Minimum : 75.3803
Maximum : 104.3583
Mean    : 89.6338
Median  : 88.5843

FULL 24-MONTH AMPI — DETRENDED RHO
     month  iss_mean_norm  sarima_norm   rho_norm  MeanPopulation_SD        SD       CV  CVPenalty  AMPI_final  AMPI_rank  AMPI_percentile
2024-01-01      83.495102   103.369885  78.050377          88.305121 13.327475 0.150925   2.011453   86.293668         17        30.434783
2024-02-01     103.445385   115.736065  96.578449         105.253300  9.705925 0.092215   0.895031  

In [6]:
# ================================================================
# AMPI FINAL VALIDATION + OUTPUT PACKAGE
# ================================================================

import pandas as pd
import numpy as np
import os

print("=" * 70)
print("AMPI FINAL VALIDATION — DE-TRENDED RHO VERSION")
print("=" * 70)

# ------------------------------------------------
# 1. Load final AMPI file
# ------------------------------------------------

input_file = "/content/AMPI_24_month_DETRENDED_RHO.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"File not found: {input_file}")

ampi = pd.read_csv(input_file)

ampi["month"] = pd.to_datetime(ampi["month"])
ampi = ampi.sort_values("month").reset_index(drop=True)

print("\nInput loaded successfully")
print("Rows:", len(ampi))
print("First month:", ampi["month"].min())
print("Last month :", ampi["month"].max())

# ------------------------------------------------
# 2. Required columns
# ------------------------------------------------

required_columns = [
    "month",
    "iss_mean_norm",
    "sarima_norm",
    "rho_norm",
    "MeanPopulation_SD",
    "SD",
    "CV",
    "CVPenalty",
    "AMPI_final"
]

missing_columns = [
    col for col in required_columns
    if col not in ampi.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns: PASSED")

# ------------------------------------------------
# 3. Number of observations
# ------------------------------------------------

assert len(ampi) == 24, "Expected exactly 24 observations"

print("24 observations: PASSED")

# ------------------------------------------------
# 4. Duplicate-month check
# ------------------------------------------------

duplicate_months = ampi["month"].duplicated().sum()

assert duplicate_months == 0, "Duplicate months found"

print("No duplicate months: PASSED")

# ------------------------------------------------
# 5. Missing-value check
# ------------------------------------------------

missing_values = ampi[required_columns].isna().sum().sum()

assert missing_values == 0, "Missing values found"

print("No missing values: PASSED")

# ------------------------------------------------
# 6. AMPI numerical validity
# ------------------------------------------------

assert np.isfinite(ampi["AMPI_final"]).all()

print("No invalid AMPI values: PASSED")

# ------------------------------------------------
# 7. Rank AMPI independently
# ------------------------------------------------

ampi["AMPI_rank"] = (
    ampi["AMPI_final"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ampi["AMPI_percentile"] = (
    (24 - ampi["AMPI_rank"]) / 23 * 100
)

# ------------------------------------------------
# 8. Independent formula check
#
# Official AMPI:
# weighted component mean - CV penalty
#
# Equal weights = 1/3 each
# ------------------------------------------------

independent_ampi = (
    (
        ampi["iss_mean_norm"]
        + ampi["sarima_norm"]
        + ampi["rho_norm"]
    ) / 3
    - ampi["CVPenalty"]
)

formula_difference = (
    independent_ampi - ampi["AMPI_final"]
).abs().max()

print("\n" + "=" * 70)
print("FORMULA VALIDATION")
print("=" * 70)

print(
    f"Maximum formula difference: "
    f"{formula_difference:.12f}"
)

assert formula_difference < 1e-10

print("OFFICIAL AMPI FORMULA: PASSED")

# ------------------------------------------------
# 9. Final statistics
# ------------------------------------------------

print("\n" + "=" * 70)
print("FINAL AMPI STATISTICS")
print("=" * 70)

print(f"Observations: {len(ampi)}")
print(f"Minimum : {ampi['AMPI_final'].min():.4f}")
print(f"Maximum : {ampi['AMPI_final'].max():.4f}")
print(f"Mean    : {ampi['AMPI_final'].mean():.4f}")
print(f"Median  : {ampi['AMPI_final'].median():.4f}")

# ------------------------------------------------
# 10. Pre-registered outage-month check
# ------------------------------------------------

outage_months = ampi[
    ampi["month"].dt.strftime("%Y-%m").isin(
        ["2025-03", "2025-04"]
    )
].copy()

print("\n" + "=" * 70)
print("PRE-REGISTERED OUTAGE MONTH CHECK")
print("=" * 70)

print(
    outage_months[
        [
            "month",
            "AMPI_final",
            "AMPI_rank",
            "AMPI_percentile",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "CVPenalty"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 11. Highest / lowest month
# ------------------------------------------------

highest = ampi.loc[
    ampi["AMPI_final"].idxmax()
]

lowest = ampi.loc[
    ampi["AMPI_final"].idxmin()
]

print("\n" + "=" * 70)
print("HIGHEST / LOWEST AMPI")
print("=" * 70)

print(
    f"Highest: {highest['month'].strftime('%Y-%m-%d')} "
    f"| AMPI = {highest['AMPI_final']:.4f}"
)

print(
    f"Lowest : {lowest['month'].strftime('%Y-%m-%d')} "
    f"| AMPI = {lowest['AMPI_final']:.4f}"
)

# ------------------------------------------------
# 12. Save final complete table
# ------------------------------------------------

final_file = "/content/AMPI_FINAL_DETRENDED_RHO_VALIDATED.csv"

ampi.to_csv(
    final_file,
    index=False
)

# ------------------------------------------------
# 13. Save outage-month check separately
# ------------------------------------------------

outage_file = "/content/AMPI_FINAL_OUTAGE_MONTH_CHECK.csv"

outage_months[
    [
        "month",
        "AMPI_final",
        "AMPI_rank",
        "AMPI_percentile",
        "iss_mean_norm",
        "sarima_norm",
        "rho_norm",
        "CVPenalty"
    ]
].to_csv(
    outage_file,
    index=False
)

# ------------------------------------------------
# 14. Final validation summary
# ------------------------------------------------

summary = pd.DataFrame({
    "metric": [
        "observations",
        "duplicate_months",
        "missing_values",
        "formula_max_difference",
        "ampi_minimum",
        "ampi_maximum",
        "ampi_mean",
        "ampi_median",
        "highest_month",
        "lowest_month",
        "march_2025_rank",
        "march_2025_percentile",
        "april_2025_rank",
        "april_2025_percentile"
    ],
    "value": [
        len(ampi),
        duplicate_months,
        missing_values,
        formula_difference,
        ampi["AMPI_final"].min(),
        ampi["AMPI_final"].max(),
        ampi["AMPI_final"].mean(),
        ampi["AMPI_final"].median(),
        highest["month"].strftime("%Y-%m-%d"),
        lowest["month"].strftime("%Y-%m-%d"),
        int(
            outage_months.loc[
                outage_months["month"].dt.strftime("%Y-%m") == "2025-03",
                "AMPI_rank"
            ].iloc[0]
        ),
        outage_months.loc[
            outage_months["month"].dt.strftime("%Y-%m") == "2025-03",
            "AMPI_percentile"
        ].iloc[0],
        int(
            outage_months.loc[
                outage_months["month"].dt.strftime("%Y-%m") == "2025-04",
                "AMPI_rank"
            ].iloc[0]
        ),
        outage_months.loc[
            outage_months["month"].dt.strftime("%Y-%m") == "2025-04",
            "AMPI_percentile"
        ].iloc[0]
    ]
})

summary_file = "/content/AMPI_FINAL_VALIDATION_SUMMARY.csv"

summary.to_csv(
    summary_file,
    index=False
)

# ------------------------------------------------
# COMPLETE
# ------------------------------------------------

print("\n" + "=" * 70)
print("FINAL VALIDATION: COMPLETE")
print("=" * 70)

print("\nFiles saved:")
print(final_file)
print(outage_file)
print(summary_file)

print("\nNo new methodology was introduced.")
print("This cell only validates and packages the corrected AMPI results.")

AMPI FINAL VALIDATION — DE-TRENDED RHO VERSION

Input loaded successfully
Rows: 24
First month: 2024-01-01 00:00:00
Last month : 2025-12-01 00:00:00

Required columns: PASSED
24 observations: PASSED
No duplicate months: PASSED
No missing values: PASSED
No invalid AMPI values: PASSED

FORMULA VALIDATION
Maximum formula difference: 0.000000000000
OFFICIAL AMPI FORMULA: PASSED

FINAL AMPI STATISTICS
Observations: 24
Minimum : 75.3803
Maximum : 104.3583
Mean    : 89.6338
Median  : 88.5843

PRE-REGISTERED OUTAGE MONTH CHECK
     month  AMPI_final  AMPI_rank  AMPI_percentile  iss_mean_norm  sarima_norm   rho_norm  CVPenalty
2025-03-01   98.442623          5        82.608696      79.651774   122.030511 107.113836   4.489417
2025-04-01   88.587671         12        52.173913      78.248056    96.672548  94.184759   1.114117

HIGHEST / LOWEST AMPI
Highest: 2024-02-01 | AMPI = 104.3583
Lowest : 2024-11-01 | AMPI = 75.3803

FINAL VALIDATION: COMPLETE

Files saved:
/content/AMPI_FINAL_DETRENDED_RH